# Module 4 — Imai 2010 causal mediation through the unknown-entity SAE latent

**v2 §3 hypothesis (after M3 retention analysis).** M3 at L21 showed a refuse → fabricate crossover at α≈0.3 with MMLU intact, but on ≈8% retention. This could be either (a) direction-specific redirection through the unknown-entity / honesty circuit Ferrando v2 identifies, or (b) selection bias on the small surviving subset. M4 tests which.

**Four arms** (Vasu 2026-05-25):
1. **A. baseline + capture** — no steering, record per-prompt SAE-latent value M(0)
2. **B. steered + capture** — desperation steering at headline α, record M(1)
3. **C. rescue** — desperation steering + clamp mediator to M(0)[qid]. If refusal recovers, the effect goes through the latent.
4. **D. reverse** — no desperation steering, instead suppress the latent directly across all prompts. If this mimics arm B's behavior, the latent alone is sufficient.

**Imai decomposition** (from A/B/C): `TE = ACME + ADE` where ACME is the indirect effect through the latent.

**Prerequisites:** M1 vectors at L21, M2 norm_scale, M3 retention table (for the α choice). Layer is fixed at 21 (Ferrando Figure 9 peak). PT SAE (gemma-scope-9b-pt-res) since entity-recognition latents are derived from the base model.

**Cost:** ≈90 min wall on A100. 4 arms × ≈20 min FaithEval + feature derivation ≈5 min. ≈$5–7 on Vast spot.

## Cell 1 — env setup

In [ ]:
from google.colab import userdata
import os
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
!git clone https://github.com/BraydenFeng/Algoverse.git /content/Algoverse 2>/dev/null || (cd /content/Algoverse && git pull)
%cd /content/Algoverse
!pip install -q -r requirements.txt sae_lens

## Cell 2 — load Gemma-2-9B-IT + PT SAE at layer 21

The IT model is what we steer; the PT SAE is where the entity-recognition latents live (Ferrando §4). The latents transfer across the IT chat finetuning per Ferrando's §5 result.

In [ ]:
import sys
sys.path.insert(0, '.')
from pathlib import Path
import torch

from src.lib.config import load_config, layer_suffix
from src.lib.model_load import load_gemma
from src.lib.sae_load import load_sae

cfg = load_config()
lsuf = layer_suffix(cfg)
layer = cfg['sae']['layer']
outputs_dir = Path(cfg['paths']['outputs_dir']) / 'm4' / lsuf
outputs_dir.mkdir(parents=True, exist_ok=True)

model, tokenizer = load_gemma(variant='primary')
print(f'loaded {model.config._name_or_path}, d_model={model.config.hidden_size}, steering+SAE layer={layer}')

sae, sae_cfg, sparsity = load_sae(layer=layer)
print(f'loaded SAE: release={cfg["sae"]["release"]}, d_sae={sae.W_dec.shape[0]}')
sae = sae.to(model.device)
sae.eval()

## Cell 3 — derive (or load) the unknown-entity latent index

Runs Ferrando §4 separation-score recipe on our hand-curated entity dataset (`src/m4_entity_dataset.py`, 60 prompts × 4 types = 240 prompts). ≈5 min on A100. Result is cached at `outputs/m4/L{layer}/unknown_entity_latents.json` — re-running this cell loads the cache instead of recomputing.

If `top_min_unknown_sep < 0.4` (the qualitative "this latent meaningfully separates" threshold), surface to Brayden — the hand-curated dataset may be too small / unbalanced and you'd need to fall back to Ferrando's full Wikidata pipeline.

In [ ]:
import json
from src.m4_feature_derivation import run_and_save, derive_unknown_entity_latent

latents_json = outputs_dir / 'unknown_entity_latents.json'
if latents_json.exists():
	print(f'loading cached {latents_json}')
	data = json.loads(latents_json.read_text())
else:
	run_and_save(model, tokenizer, sae)
	data = json.loads(latents_json.read_text())

top_latent_idx = data['top_unknown_entity_latent_idx']
top_sep = data['top_min_unknown_sep']
print(f'\ntop unknown-entity latent: idx={top_latent_idx}, min-across-types sep={top_sep:.3f}')
if top_sep < 0.4:
	print('** WARN: separation score is weak. Surface to Brayden before running mediation arms.')
print('top-10 candidates:')
for entry in data['top_n']:
	print(f'  idx={entry["idx"]:5d}  sep={entry["min_unknown_sep"]:+.3f}')

## Cell 4 — estimate residual norm for steering

Same calibration as M2/M3 — mean ||h|| at L21 on the neutral corpus. Needed for the steering hook used by arms B and C.

In [ ]:
import numpy as np
from src.steering import load_emotion_vector, estimate_residual_norm
from src.extract_vectors import _load_neutral_corpus

desperation = load_emotion_vector('desperation')
print(f'desperation vector loaded: shape={desperation.shape}, ||v||={np.linalg.norm(desperation):.4f}')

neutral_texts = _load_neutral_corpus(Path(cfg['paths']['data_dir']))
norm_scale = estimate_residual_norm(model, tokenizer, layer=layer, calibration_texts=neutral_texts)
print(f'norm_scale @ L{layer} = {norm_scale:.2f}')

# headline alpha for M4 — the L21 M3 crossover lives at α≈0.3, MMLU still intact
ALPHA_HEAD = 0.3

## Cell 5 — helpers (per-prompt run + mediator capture + classification)

Thin wrapper that runs FaithEval prompt-by-prompt, applying a per-prompt hook factory (so the rescue arm can use per-prompt clamp values from arm A). Records mediator value, generated output, and classifier label.

In [ ]:
import pandas as pd
from dataclasses import asdict, dataclass
from datasets import load_dataset
from tqdm import tqdm

from src.lib.classifier import classify
from src.m4_mediation import _last_prompt_token_idx

@dataclass
class M4Record:
	qid: str
	question: str
	context: str
	output: str
	label: str
	method: str
	reason: str
	mediator: float

ds = load_dataset(cfg['faitheval']['hf_dataset'], split='test')
template = cfg['faitheval']['prompt_template']
print(f'FaithEval: {len(ds)} prompts')

def _build_prompt(context, question):
	return template.format(context=context, question=question)

def run_arm(
	arm_name,
	hook_factory_fn,  # callable(qid, prompt, target_tok_idx) -> hook_factory
	checkpoint_path,
	limit=None,
	checkpoint_every=500,
):
	records = []
	resume_qids = set()
	if checkpoint_path.exists():
		prev = pd.read_csv(checkpoint_path)
		records = [M4Record(**r) for r in prev.to_dict(orient='records')]
		resume_qids = set(prev['qid'].tolist())
		print(f'[{arm_name}] resumed from {checkpoint_path}: {len(resume_qids)} done')

	ds_iter = ds if limit is None else ds.select(range(min(limit, len(ds))))
	for i, row in enumerate(tqdm(ds_iter, desc=arm_name)):
		qid = row['qid']
		if qid in resume_qids:
			continue
		prompt = _build_prompt(row['context'], row['question'])
		target_tok = _last_prompt_token_idx(tokenizer, prompt)

		capture = {'value': float('nan')}
		factory = hook_factory_fn(qid, prompt, target_tok, capture)
		handle = factory(model) if factory is not None else None
		try:
			enc = tokenizer(prompt, return_tensors='pt').to(model.device)
			with torch.no_grad():
				out = model.generate(
					**enc, max_new_tokens=128, do_sample=False,
					pad_token_id=tokenizer.eos_token_id,
				)
			generated = out[0][enc['input_ids'].shape[1]:]
			output_text = tokenizer.decode(generated, skip_special_tokens=True).strip()
		except Exception as e:
			output_text = ''
			print(f'[{arm_name}] gen failed qid={qid}: {e}')
		finally:
			if handle is not None:
				handle.remove()

		result = classify(output_text, row['question'], row['context'])
		records.append(M4Record(
			qid=qid, question=row['question'], context=row['context'],
			output=output_text, label=result.label, method=result.method,
			reason=result.reason, mediator=capture['value'],
		))

		if (i + 1) % checkpoint_every == 0:
			pd.DataFrame([asdict(r) for r in records]).to_csv(checkpoint_path, index=False)

	df = pd.DataFrame([asdict(r) for r in records])
	df.to_csv(checkpoint_path, index=False)
	return df

def _rates(df):
	df = df.copy()
	df['output'] = df['output'].fillna('').astype(str)
	ne = df[df['output'].str.strip() != '']
	n = max(len(ne), 1)
	return {
		'refuse': float((ne['label'] == 'refuses').sum() / n),
		'fabricate': float((ne['label'] == 'fabricates').sum() / n),
		'off_topic': float((ne['label'] == 'off_topic').sum() / n),
		'n_nonempty': len(ne),
		'n_total': len(df),
	}

## Cell 6 — Arm A: baseline + mediator capture

No intervention; just capture the unknown-entity latent's value at the last prompt token. ≈20 min.

In [ ]:
from src.m4_mediation import make_mediator_capture_hook

def arm_a_factory(qid, prompt, target_tok, capture):
	return make_mediator_capture_hook(sae, top_latent_idx, layer, target_tok, capture)

df_A = run_arm('arm_A_baseline', arm_a_factory, outputs_dir / 'arm_A.csv')
rates_A = _rates(df_A)
print('A:', rates_A)
print('mediator stats:', df_A['mediator'].describe()[['mean', '50%', 'min', 'max']].to_dict())

## Cell 7 — Arm B: steered + mediator capture

Desperation steering at α=0.3, recording mediator. ≈20 min.

In [ ]:
from src.steering import make_steering_hook_factory

steering_factory_template = make_steering_hook_factory(
	vector=desperation, layer=layer, alpha=ALPHA_HEAD, norm_scale=norm_scale,
)

def arm_b_factory(qid, prompt, target_tok, capture):
	# steering AND capture: chain by composing in one hook
	# steering writes; capture reads after steering is applied
	v_unit = (torch.tensor(desperation, dtype=model.dtype, device=model.device)
	          / torch.tensor(desperation, dtype=model.dtype, device=model.device).norm())
	steering_vec = v_unit * float(ALPHA_HEAD * norm_scale)

	def factory(m):
		def hook(module, args, output):
			h = output[0] if isinstance(output, tuple) else output
			h_steered = h + steering_vec
			if h_steered.shape[1] > target_tok:
				vec = h_steered[0, target_tok, :].float()
				with torch.no_grad():
					a = sae.encode(vec.unsqueeze(0).to(next(sae.parameters()).dtype)).squeeze(0)
				capture['value'] = float(a[top_latent_idx].item())
			if isinstance(output, tuple):
				return (h_steered, *output[1:])
			return h_steered
		return m.model.layers[layer].register_forward_hook(hook)
	return factory

df_B = run_arm('arm_B_steered', arm_b_factory, outputs_dir / 'arm_B.csv')
rates_B = _rates(df_B)
print('B:', rates_B)
print('mediator stats:', df_B['mediator'].describe()[['mean', '50%', 'min', 'max']].to_dict())

## Cell 8 — Arm C: rescue (steered + clamp mediator to per-prompt baseline)

For each prompt, look up A's captured mediator value and clamp the latent to that during the steered forward pass. If refusal recovers toward baseline, the steering effect goes through the latent. ≈20 min.

In [ ]:
from src.m4_mediation import make_rescue_hook

# per-prompt clamp values: from arm A's captured mediator
clamp_lookup = dict(zip(df_A['qid'], df_A['mediator']))

v_unit = (torch.tensor(desperation, dtype=model.dtype, device=model.device)
          / torch.tensor(desperation, dtype=model.dtype, device=model.device).norm())
steering_vec_dev = v_unit * float(ALPHA_HEAD * norm_scale)

def arm_c_factory(qid, prompt, target_tok, capture):
	clamp_value = float(clamp_lookup.get(qid, float('nan')))
	if not np.isfinite(clamp_value):
		return None  # arm A didn't capture this qid; skip
	return make_rescue_hook(
		sae=sae, latent_idx=top_latent_idx, layer=layer,
		target_token_idx=target_tok, clamp_value=clamp_value,
		steering_vector_on_device=steering_vec_dev,
	)

df_C = run_arm('arm_C_rescue', arm_c_factory, outputs_dir / 'arm_C.csv')
rates_C = _rates(df_C)
print('C:', rates_C)

## Cell 9 — Arm D: reverse (suppress latent directly, no desperation steering)

Vasu's request: directly suppress the unknown-entity latent (set it low across all prompts) without desperation steering. If this reproduces arm B's behavior, the latent alone is sufficient to drive refusal→fabrication. Clamp value = 10th percentile of arm A baseline. ≈20 min.

In [ ]:
from src.m4_mediation import make_mediator_clamp_hook

low_clamp = float(np.nanpercentile(df_A['mediator'], 10))
print(f'suppression clamp value (10th percentile of A): {low_clamp:.4f}')

# arm D uses make_mediator_clamp_hook with no steering — simpler than rescue
# (no chained steering, only the clamp). The existing helper takes
# desperation_steering_factory=None which raises NotImplementedError; bypass by
# building the clamp hook inline since we don't need steering composition.
W_dec_j = sae.W_dec[top_latent_idx].detach().to(model.device).to(model.dtype)

def arm_d_factory(qid, prompt, target_tok, capture):
	def factory(m):
		def hook(module, args, output):
			h = output[0] if isinstance(output, tuple) else output
			if h.shape[1] > target_tok:
				vec = h[0, target_tok, :].float()
				with torch.no_grad():
					a = sae.encode(vec.unsqueeze(0).to(next(sae.parameters()).dtype)).squeeze(0)
				a_j = float(a[top_latent_idx].item())
				delta = (low_clamp - a_j) * W_dec_j
				h_new = h.clone()
				h_new[0, target_tok, :] = h_new[0, target_tok, :] + delta
				capture['value'] = low_clamp  # we forced it
				if isinstance(output, tuple):
					return (h_new, *output[1:])
				return h_new
			return output
		return m.model.layers[layer].register_forward_hook(hook)
	return factory

df_D = run_arm('arm_D_reverse', arm_d_factory, outputs_dir / 'arm_D.csv')
rates_D = _rates(df_D)
print('D:', rates_D)

## Cell 10 — aggregate, decompose, upload

Computes Imai TE/ACME/ADE from arms A, B, C. Arm D is informational (does suppression alone mimic steering?).

In [ ]:
from huggingface_hub import upload_file
from src.m4_mediation import compute_mediation, format_result

rates = {'A': rates_A, 'B': rates_B, 'C': rates_C, 'D': rates_D}
n_per_arm = {arm: rates[arm]['n_nonempty'] for arm in rates}
mediator_stats = {
	'A_mean': float(df_A['mediator'].mean()),
	'B_mean': float(df_B['mediator'].mean()),
	'A_to_B_shift': float(df_B['mediator'].mean() - df_A['mediator'].mean()),
}

result = compute_mediation(rates, n_per_arm, mediator_stats)
decision = format_result(result)
decision += '\n\nReverse arm (Vasu): does latent suppression alone mimic desperation steering?\n'
decision += f'  arm B (steered)    refuse={rates_B["refuse"]:.3f}\n'
decision += f'  arm D (suppression) refuse={rates_D["refuse"]:.3f}\n'
decision += f'  arm A (baseline)   refuse={rates_A["refuse"]:.3f}\n'
print(decision)

(outputs_dir / 'decision.txt').write_text(decision, encoding='utf-8')

# upload all arm CSVs + decision via upload_folder (1 commit per file, but we
# already paid the rate-limit cost during M3; budget here is small)
for arm in 'ABCD':
	p = outputs_dir / f'arm_{arm}.csv'
	if p.exists():
		try:
			upload_file(path_or_fileobj=str(p),
			            path_in_repo=f'm4/{lsuf}/arm_{arm}.csv',
			            repo_id=cfg['paths']['hf_artifact_repo'], repo_type='dataset')
			print(f'synced arm_{arm}.csv')
		except Exception as e:
			print(f'sync arm_{arm}.csv failed (non-fatal): {e}')
try:
	upload_file(path_or_fileobj=str(outputs_dir / 'decision.txt'),
	            path_in_repo=f'm4/{lsuf}/decision.txt',
	            repo_id=cfg['paths']['hf_artifact_repo'], repo_type='dataset')
	print('synced decision.txt')
except Exception as e:
	print(f'sync decision.txt failed (non-fatal): {e}')

## Interpretation guide (human-owned)

- **|ACME| / |TE| close to 1** — the latent fully mediates desperation's effect on refusal. Headline positive result.
- **|ACME| / |TE| close to 0** — the latent doesn't mediate; the steering effect bypasses it. Combined with the M3 retention finding, this points to direction-non-specific perturbation as the mechanism.
- **|ACME| / |TE| around 0.3–0.6** — partial mediation. Publishable as "the latent is one route, not the only route."
- **Arm D refusal close to arm B's** — latent suppression alone reproduces desperation steering. Strong evidence the latent is the causal lever.
- **Arm D refusal close to arm A's** — suppression alone does nothing; latent isn't sufficient.

The choice between these framings is Brayden's. The agent should report numbers and the suggested interpretation tag; the paper-level call is human-owned per CLAUDE.md.